In [2]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer

# Download required NLTK resources for tokenization, stopwords, and lemmatization
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)

print("Libraries imported and NLTK resources configured successfully.")

Libraries imported and NLTK resources configured successfully.


# Loading Text Dataset and Performing Text Preprocessing

In [3]:
# Sample text corpus for text processing and feature extraction
documents = [
    "Data science involves studying data, machine learning models, and algorithms.",
    "Machine learning algorithms learn patterns from training data to make predictions.",
    "Natural language processing helps machines understand human languages and text.",
    "Data preprocessing and feature scaling improve machine learning model performance."
]

# Set of English stopwords
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase the text
    text = text.lower()
    # 2. Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)
    # 3. Tokenize words
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    filtered_tokens = [w for w in tokens if w not in stop_words]
    return " ".join(filtered_tokens)

cleaned_documents = [preprocess_text(doc) for doc in documents]

# Display original vs preprocessed text
for i, (orig, clean) in enumerate(zip(documents, cleaned_documents), 1):
    print(f"Doc {i} Original: {orig}")
    print(f"Doc {i} Cleaned : {clean}\n")

Doc 1 Original: Data science involves studying data, machine learning models, and algorithms.
Doc 1 Cleaned : data science involves studying data machine learning models algorithms

Doc 2 Original: Machine learning algorithms learn patterns from training data to make predictions.
Doc 2 Cleaned : machine learning algorithms learn patterns training data make predictions

Doc 3 Original: Natural language processing helps machines understand human languages and text.
Doc 3 Cleaned : natural language processing helps machines understand human languages text

Doc 4 Original: Data preprocessing and feature scaling improve machine learning model performance.
Doc 4 Cleaned : data preprocessing feature scaling improve machine learning model performance



# Applying TF-IDF Vectorization and Displaying Transformed Output

In [4]:
# TF-IDF Vectorization on the preprocessed text dataset
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_documents)

# Create DataFrame for tabular representation of TF-IDF feature matrix
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(cleaned_documents))]
)

print(f"TF-IDF Vocabulary Size: {len(tfidf_vectorizer.get_feature_names_out())}")
print("TF-IDF Feature Matrix:")
tfidf_df

TF-IDF Vocabulary Size: 28
TF-IDF Feature Matrix:


,algorithms,data,feature,helps,human,improve,involves,language,languages,learn,...,performance,predictions,preprocessing,processing,scaling,science,studying,text,training,understand
Doc 1,0.296595,0.480239,0.000000,0.000000,0.000000,0.000000,0.376194,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.376194,0.376194,0.000000,0.000000,0.000000
Doc 2,0.301372,0.243987,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.382253,...,0.000000,0.382253,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.382253,0.000000
Doc 3,0.000000,0.000000,0.000000,0.333333,0.333333,0.000000,0.000000,0.333333,0.333333,0.000000,...,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.000000,0.333333,0.000000,0.333333
Doc 4,0.000000,0.237509,0.372104,0.000000,0.000000,0.372104,0.000000,0.000000,0.000000,0.000000,...,0.372104,0.000000,0.372104,0.000000,0.372104,0.000000,0.000000,0.000000,0.000000,0.000000


## 1. Apply CountVectorizer to the same text dataset

In [5]:
# Applying CountVectorizer (Bag of Words) to the same preprocessed text dataset
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(cleaned_documents)

# Create DataFrame for tabular representation of CountVectorizer matrix
count_df = pd.DataFrame(
    count_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(cleaned_documents))]
)

print(f"CountVectorizer Vocabulary Size: {len(count_vectorizer.get_feature_names_out())}")
print("CountVectorizer (Bag of Words) Matrix:")
count_df

CountVectorizer Vocabulary Size: 28
CountVectorizer (Bag of Words) Matrix:


,algorithms,data,feature,helps,human,improve,involves,language,languages,learn,...,performance,predictions,preprocessing,processing,scaling,science,studying,text,training,understand
Doc 1,1,2,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,1,0,0,0
Doc 2,1,1,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,1,0
Doc 3,0,0,0,1,1,0,0,1,1,0,...,0,0,0,1,0,0,0,1,0,1
Doc 4,0,1,1,0,0,1,0,0,0,0,...,1,0,1,0,1,0,0,0,0,0


## 2. Compare TF-IDF and CountVectorizer outputs

In [6]:
# 1. Term-by-Term Comparison for Document 1
doc1_comparison = pd.DataFrame({
    'Word': count_vectorizer.get_feature_names_out(),
    'CountVectorizer (Term Frequency)': count_matrix.toarray()[0],
    'TF-IDF Score': np.round(tfidf_matrix.toarray()[0], 4)
})

# Display words present in Doc 1
print("--- Comparison of CountVectorizer vs TF-IDF for Doc 1 Terms ---")
display(doc1_comparison[doc1_comparison['CountVectorizer (Term Frequency)'] > 0].reset_index(drop=True))

# 2. Summary of Key Differences
comparison_summary = pd.DataFrame({
    'Feature': [
        'Output Values',
        'Word Importance Weighting',
        'Impact of Frequent Words',
        'Length Normalization',
        'Primary Application'
    ],
    'CountVectorizer': [
        'Integer counts (raw term frequency: 1, 2, 3...)',
        'Equal weight to all occurrences based solely on count',
        'Common corpus words dominate due to high frequencies',
        'Not normalized (longer documents have higher counts)',
        'Basic text modeling, frequency counts, simple classification'
    ],
    'TF-IDF Vectorizer': [
        'Floating-point scores between 0.0 and 1.0',
        'Penalizes words frequent across multiple documents (IDF)',
        'Domain-specific rare words receive higher significance weights',
        'Applies Euclidean (L2) normalization by default',
        'Information retrieval, document search, text classification'
    ]
})

print("\n--- Summary Comparison Between CountVectorizer and TF-IDF ---")
display(comparison_summary)

--- Comparison of CountVectorizer vs TF-IDF for Doc 1 Terms ---


,Word,CountVectorizer (Term Frequency),TF-IDF Score
0,algorithms,1,0.2966
1,data,2,0.4802
2,involves,1,0.3762
3,learning,1,0.2401
4,machine,1,0.2401
5,models,1,0.3762
6,science,1,0.3762
7,studying,1,0.3762



--- Summary Comparison Between CountVectorizer and TF-IDF ---


,Feature,CountVectorizer,TF-IDF Vectorizer
0,Output Values,"Integer counts (raw term frequency: 1, 2, 3...)",Floating-point scores between 0.0 and 1.0
1,Word Importance Weighting,Equal weight to all occurrences based solely o...,Penalizes words frequent across multiple docum...
2,Impact of Frequent Words,Common corpus words dominate due to high frequ...,Domain-specific rare words receive higher sign...
3,Length Normalization,Not normalized (longer documents have higher c...,Applies Euclidean (L2) normalization by default
4,Primary Application,"Basic text modeling, frequency counts, simple ...","Information retrieval, document search, text c..."


## 3. Perform lemmatization instead of stemming

In [7]:
# Initializing Porter Stemmer and WordNet Lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Representative words extracted from the dataset
sample_words = [
    "studying", "studies", "learning", "algorithms",
    "patterns", "predictions", "languages", "scaling", "improves"
]

records = []
for word in sample_words:
    stemmed = stemmer.stem(word)
    lemmatized_noun = lemmatizer.lemmatize(word, pos='n')
    lemmatized_verb = lemmatizer.lemmatize(word, pos='v')
    records.append({
        'Original Word': word,
        'Porter Stemmer': stemmed,
        'Lemmatizer (Noun)': lemmatized_noun,
        'Lemmatizer (Verb)': lemmatized_verb
    })

comparison_df = pd.DataFrame(records)
print("--- Comparison of Stemming vs Lemmatization on Sample Words ---")
display(comparison_df)

# Applying Lemmatization to the preprocessed dataset
def lemmatize_text(text):
    tokens = word_tokenize(text)
    return " ".join([lemmatizer.lemmatize(w) for w in tokens])

lemmatized_documents = [lemmatize_text(doc) for doc in cleaned_documents]

print("\n--- Preprocessed vs Lemmatized Documents ---")
for i, (clean, lem) in enumerate(zip(cleaned_documents, lemmatized_documents), 1):
    print(f"Doc {i} Preprocessed: {clean}")
    print(f"Doc {i} Lemmatized  : {lem}\n")

--- Comparison of Stemming vs Lemmatization on Sample Words ---


,Original Word,Porter Stemmer,Lemmatizer (Noun),Lemmatizer (Verb)
0,studying,studi,studying,study
1,studies,studi,study,study
2,learning,learn,learning,learn
3,algorithms,algorithm,algorithm,algorithms
4,patterns,pattern,pattern,pattern
5,predictions,predict,prediction,predictions
6,languages,languag,language,languages
7,scaling,scale,scaling,scale
8,improves,improv,improves,improve



--- Preprocessed vs Lemmatized Documents ---
Doc 1 Preprocessed: data science involves studying data machine learning models algorithms
Doc 1 Lemmatized  : data science involves studying data machine learning model algorithm

Doc 2 Preprocessed: machine learning algorithms learn patterns training data make predictions
Doc 2 Lemmatized  : machine learning algorithm learn pattern training data make prediction

Doc 3 Preprocessed: natural language processing helps machines understand human languages text
Doc 3 Lemmatized  : natural language processing help machine understand human language text

Doc 4 Preprocessed: data preprocessing feature scaling improve machine learning model performance
Doc 4 Lemmatized  : data preprocessing feature scaling improve machine learning model performance



## 4. Apply Min-Max Scaling to another numerical dataset

In [8]:
# Creating another numerical dataset with features of varying scales
numerical_data = pd.DataFrame({
    'Age': [22, 25, 47, 52, 36, 29, 61],
    'Salary': [25000, 32000, 85000, 110000, 62000, 48000, 135000],
    'Experience_Years': [1, 3, 20, 25, 12, 5, 32],
    'Credit_Score': [600, 650, 780, 810, 710, 680, 840]
})

print("--- Original Numerical Dataset ---")
display(numerical_data)

# Applying Min-Max Scaling: X_scaled = (X - X_min) / (X_max - X_min)
min_max_scaler = MinMaxScaler()
scaled_data = min_max_scaler.fit_transform(numerical_data)

min_max_df = pd.DataFrame(
    np.round(scaled_data, 4),
    columns=numerical_data.columns,
    index=numerical_data.index
)

print("\n--- Min-Max Scaled Dataset (Bounded within [0, 1]) ---")
display(min_max_df)

print("\n--- Summary Statistics of Min-Max Scaled Data ---")
display(min_max_df.describe().loc[['min', 'max', 'mean', 'std']])

--- Original Numerical Dataset ---


,Age,Salary,Experience_Years,Credit_Score
0,22,25000,1,600
1,25,32000,3,650
2,47,85000,20,780
3,52,110000,25,810
4,36,62000,12,710
5,29,48000,5,680
6,61,135000,32,840



--- Min-Max Scaled Dataset (Bounded within [0, 1]) ---


,Age,Salary,Experience_Years,Credit_Score
0,0.0000,0.0000,0.0000,0.0000
1,0.0769,0.0636,0.0645,0.2083
2,0.6410,0.5455,0.6129,0.7500
3,0.7692,0.7727,0.7742,0.8750
4,0.3590,0.3364,0.3548,0.4583
5,0.1795,0.2091,0.1290,0.3333
6,1.0000,1.0000,1.0000,1.0000



--- Summary Statistics of Min-Max Scaled Data ---


,Age,Salary,Experience_Years,Credit_Score
min,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000
mean,0.432229,0.418186,0.419343,0.517843
std,0.378993,0.372248,0.385309,0.368672


## 5. Compare Normalization and Standardization results

In [9]:
# 1. Standardization (StandardScaler: mean=0, std=1)
standard_scaler = StandardScaler()
standardized_values = standard_scaler.fit_transform(numerical_data)
standardized_df = pd.DataFrame(
    np.round(standardized_values, 4),
    columns=numerical_data.columns,
    index=numerical_data.index
)

# 2. Normalization (Normalizer: L2 norm per sample row = 1)
normalizer = Normalizer(norm='l2')
normalized_values = normalizer.fit_transform(numerical_data)
normalized_df = pd.DataFrame(
    np.round(normalized_values, 6),
    columns=numerical_data.columns,
    index=numerical_data.index
)

print("--- 1. Original Dataset Statistics ---")
display(numerical_data.describe().loc[['mean', 'std', 'min', 'max']])

print("\n--- 2. Standardization (Z-Score: mean ≈ 0, std = 1) ---")
display(standardized_df)
print("Standardized Statistics:")
display(standardized_df.describe().loc[['mean', 'std', 'min', 'max']])

print("\n--- 3. Normalization (L2 Norm: sample vector length = 1) ---")
display(normalized_df)
print("L2 Norm per row (verification that each sample has unit length = 1.0):")
print(np.round(np.linalg.norm(normalized_values, axis=1), 4))

# 4. Comparison Summary
comparison_table = pd.DataFrame({
    'Aspect': [
        'Mathematical Formula',
        'Resulting Range',
        'Mean & Standard Deviation',
        'Axis of Operation',
        'Outlier Sensitivity',
        'Best Suited Machine Learning Algorithms'
    ],
    'Standardization (StandardScaler)': [
        '(X - μ) / σ',
        'Unbounded (commonly between -3 and +3)',
        'Mean = 0, Standard Deviation = 1',
        'Column-wise (operates feature by feature)',
        'Robust to outliers (preserves relative variance without clipping)',
        'Linear/Logistic Regression, SVM, PCA, Linear Discriminant Analysis'
    ],
    'Normalization (Normalizer)': [
        'X / ||X||_2  (or Min-Max: (X - min) / (max - min))',
        'L2: Unit hypersphere (norm = 1) | Min-Max: [0, 1]',
        'Varies depending on input magnitudes',
        'Row-wise (operates sample by sample in L2) / Column-wise (Min-Max)',
        'Sensitive to outliers (Min-Max compresses range; L2 alters vector angles)',
        'k-Nearest Neighbors (k-NN), K-Means clustering, Cosine Similarity, ANN'
    ]
})

print("\n--- 4. Comparison Summary: Normalization vs Standardization ---")
display(comparison_table)

--- 1. Original Dataset Statistics ---


,Age,Salary,Experience_Years,Credit_Score
mean,38.857143,71000.000000,14.000000,724.285714
std,14.780940,40947.120371,11.944315,88.479215
min,22.000000,25000.000000,1.000000,600.000000
max,61.000000,135000.000000,32.000000,840.000000



--- 2. Standardization (Z-Score: mean ≈ 0, std = 1) ---


,Age,Salary,Experience_Years,Credit_Score
0,-1.2318,-1.2134,-1.1756,-1.5172
1,-1.0126,-1.0288,-0.9947,-0.9069
2,0.5950,0.3693,0.5426,0.6801
3,0.9604,1.0288,0.9947,1.0464
4,-0.2088,-0.2374,-0.1809,-0.1744
5,-0.7203,-0.6067,-0.8139,-0.5406
6,1.6181,1.6882,1.6277,1.4126


Standardized Statistics:


,Age,Salary,Experience_Years,Credit_Score
mean,6.344132e-17,-3.172066e-17,-0.000014,-3.172066e-17
std,1.080105e+00,1.080127e+00,1.080114,1.080120e+00
min,-1.231800e+00,-1.213400e+00,-1.175600,-1.517200e+00
max,1.618100e+00,1.688200e+00,1.627700,1.412600e+00



--- 3. Normalization (L2 Norm: sample vector length = 1) ---


,Age,Salary,Experience_Years,Credit_Score
0,0.000880,0.999712,0.000040,0.023993
1,0.000781,0.999793,0.000094,0.020308
2,0.000553,0.999958,0.000235,0.009176
3,0.000473,0.999973,0.000227,0.007363
4,0.000581,0.999934,0.000194,0.011451
5,0.000604,0.999899,0.000104,0.014165
6,0.000452,0.999981,0.000237,0.006222


L2 Norm per row (verification that each sample has unit length = 1.0):
[1. 1. 1. 1. 1. 1. 1.]

--- 4. Comparison Summary: Normalization vs Standardization ---


,Aspect,Standardization (StandardScaler),Normalization (Normalizer)
0,Mathematical Formula,(X - μ) / σ,X / ||X||_2 (or Min-Max: (X - min) / (max - m...
1,Resulting Range,Unbounded (commonly between -3 and +3),"L2: Unit hypersphere (norm = 1) | Min-Max: [0, 1]"
2,Mean & Standard Deviation,"Mean = 0, Standard Deviation = 1",Varies depending on input magnitudes
3,Axis of Operation,Column-wise (operates feature by feature),Row-wise (operates sample by sample in L2) / C...
4,Outlier Sensitivity,Robust to outliers (preserves relative varianc...,Sensitive to outliers (Min-Max compresses rang...
5,Best Suited Machine Learning Algorithms,"Linear/Logistic Regression, SVM, PCA, Linear D...","k-Nearest Neighbors (k-NN), K-Means clustering..."
